In [ ]:
import sys, importlib, os

_src_training = os.path.dirname(os.path.abspath("training.ipynb"))
_src = os.path.dirname(_src_training)
_project_root = os.path.dirname(_src)
for _p in [_src_training, _src, _project_root, os.path.join(_project_root, "common")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import infrastructure, pipeline, classifier, evaluation, concept_visualization
for _mod in [infrastructure, pipeline, classifier, evaluation, concept_visualization]:
    importlib.reload(_mod)

from infrastructure import clean_neo4j_db, clean_kafka_topics, delete_test_neo4j_nodes, verify_concept_created
from pipeline import train_mnist, remove_concept, retrain_concept
from evaluation import test_mnist_all
from classifier import classify_image
from concept_visualization import load_concepts, plot_concepts, restore_concepts_to_neo4j
import json, uuid
print("Modules loaded.")

In [2]:
classes_to_subclasses = {
    0: [1],
    1: [1, 3],
    2: [1, 2],
    3: [1],
    4: [1, 2],
    5: [1],
    6: [1],
    7: [1],
    8: [1],
    9: [2],
}

In [3]:
#clean_neo4j_db()
#clean_kafka_topics()

#for class_num in classes_to_subclasses:
#    for subclass in classes_to_subclasses[class_num]:
#        train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_concept_creation=True)

In [ ]:
# ── Restore concepts from JSON snapshot into Neo4j ──
# Uncomment and run to load colleague's concepts (replaces existing ones)

# restore_concepts_to_neo4j("../../concept_graphs.json")

In [ ]:
# ── Visualize concept graphs ──
# Use "neo4j" to load from database, or a file path for JSON snapshot
concepts = load_concepts("neo4j")
# concepts = load_concepts("../../concept_graphs.json")
plot_concepts(concepts)

In [ ]:
# ── Test on generated_samples with all 3 optimized param sets ──
from sklearn.metrics import accuracy_score

# --- Best FGW params (CMA-ES, 500 trials) ---
params_fgw_best = {
    "comparison_method": "fgw",
    "fgw_alpha": 0.7548,
    "ged_timeout": 12.14,
    "skeletonization_threshold": 150,
    "simplification_epsilon": 3.52,
    "features": ["normalized_x", "normalized_y", "horizontal_direction", "cycle_count", "angle_with_ox"],
    "property_normalizers": {
        "normalized_x": 2.83, "normalized_y": 3.03,
        "horizontal_direction": 1.25, "vertical_direction": 1.68,
        "cycle_count": 3.81, "angle_with_ox": 175.50,
    },
    "node_costs": {
        "NO_COST": 0.0, "MINOR": 0.076, "GENERAL": 0.246,
        "SEVERE": 0.383, "NO_MATCH": 1.623, "IMPOSSIBLE": 23.975,
    },
}

# --- Best GED params (CMA-ES, 500 trials) ---
params_ged_best = {
    "comparison_method": "ged",
    "ged_timeout": 6.25,
    "skeletonization_threshold": 110,
    "simplification_epsilon": 4.62,
    "features": ["normalized_x", "normalized_y", "horizontal_direction", "cycle_count", "angle_with_ox"],
    "property_normalizers": {
        "normalized_x": 3.38, "normalized_y": 2.26,
        "horizontal_direction": 2.70, "vertical_direction": 3.06,
        "cycle_count": 2.66, "angle_with_ox": 252.33,
    },
    "node_costs": {
        "NO_COST": 0.0, "MINOR": 0.230, "GENERAL": 0.468,
        "SEVERE": 0.754, "NO_MATCH": 1.181, "IMPOSSIBLE": 7.378,
    },
}

# --- Best GED narrowed params (CMA-ES, 200 trials, 78.4% proxy) ---
params_ged_narrowed = {
    "comparison_method": "ged",
    "ged_timeout": 8.72,
    "skeletonization_threshold": 140,
    "simplification_epsilon": 4.55,
    "features": ["normalized_x", "normalized_y", "horizontal_direction", "cycle_count", "angle_with_ox"],
    "property_normalizers": {
        "normalized_x": 3.75, "normalized_y": 3.67,
        "horizontal_direction": 3.11, "vertical_direction": 3.0,
        "cycle_count": 3.28, "angle_with_ox": 279.36,
    },
    "node_costs": {
        "NO_COST": 0.0, "MINOR": 0.254, "GENERAL": 0.449,
        "SEVERE": 0.725, "NO_MATCH": 1.304, "IMPOSSIBLE": 6.681,
    },
}

all_params = {
    "FGW best": params_fgw_best,
    "GED best": params_ged_best,
    "GED narrowed": params_ged_narrowed,
}

summary = []
for name, params in all_params.items():
    print(f"\n{'='*60}")
    print(f"Running: {name} ({params['comparison_method']})")
    print(f"{'='*60}")
    r, yt, yp, rd = test_mnist_all(
        classes=list(classes_to_subclasses.keys()),
        params=params,
        sample_fraction=1.0,
        description=f"generated_samples — {name}",
    )
    acc = accuracy_score(yt, yp)
    summary.append((name, acc, rd))

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for name, acc, rd in summary:
    print(f"  {name:20s}  accuracy: {acc*100:.2f}%")


In [18]:
# ── Experiment: Test on MNIST-all (10k images from datasets/mnist_all/) ──
# Copies images into tests/ so nuclio containers can access them via volume mount
import os, pathlib, shutil

_project_root = pathlib.Path(os.path.abspath("training.ipynb")).parent.parent.parent
_src_dir = _project_root / "datasets" / "mnist_all"
_dst_dir = _project_root / "tests" / "mnist_all"

# Remove old symlink if it exists, then copy
if _dst_dir.is_symlink():
    _dst_dir.unlink()

if not _dst_dir.exists():
    print(f"Copying {_src_dir} -> {_dst_dir} ...")
    shutil.copytree(_src_dir, _dst_dir)
    print(f"Done. Copied {sum(1 for _ in _dst_dir.rglob('*.png'))} images.")
    
else:
    print(f"Directory already exists: {_dst_dir}")

params_mnist_all = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}

results_mnist_all, y_true_mnist_all, y_pred_mnist_all, run_dir_mnist_all = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params_ged_narrowed,
    sample_fraction=1.0,
    description="Experiment: MNIST-all 10k test set from datasets/mnist_all",
    local_path_template=str(_dst_dir / "{cls}"),
    nuclio_volume_path_template="/opt/nuclio/shared_storage/mnist_all/{cls}",
)

In [21]:
# ── Experiment: Test on manifest "complete" images only ──
import os, pathlib, shutil
import pandas as pd
from PIL import Image

_project_root = pathlib.Path(os.path.abspath("training.ipynb")).parent.parent.parent
_datasets_dir = _project_root / "datasets"
_manifest = pd.read_csv(_datasets_dir / "mnist_all_manifest.csv")

# Filter for complete images only
_complete = _manifest[_manifest["structure"] == "complete"].copy()
print(f"Total 'complete' images in manifest: {len(_complete)}")
print(f"Per class:\n{_complete['class'].value_counts().sort_index().to_string()}")

# Build a filtered directory with copies: tests/manifest_complete/{cls}/
_filtered_root = _project_root / "tests" / "manifest_complete"
if _filtered_root.exists():
    shutil.rmtree(_filtered_root)

copied = 0
for _, row in _complete.iterrows():
    cls = int(row["class"])
    src_image = _datasets_dir / row["image_path"]
    if not src_image.exists():
        continue
    dst_dir = _filtered_root / str(cls)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_file = dst_dir / src_image.name
    if not dst_file.exists():
        shutil.copy2(src_image, dst_file)
        copied += 1

# Resize all images to 100x100 (concepts were trained on this resolution)
resized = 0
for cls_dir in sorted(_filtered_root.iterdir()):
    if not cls_dir.is_dir():
        continue
    for img_path in cls_dir.glob("*.png"):
        img = Image.open(img_path)
        if img.size != (100, 100):
            img.resize((100, 100), Image.LANCZOS).save(img_path)
            resized += 1

print(f"Copied {copied} images, resized {resized} to 100x100")
for d in sorted(_filtered_root.iterdir()):
    if d.is_dir():
        print(f"  Class {d.name}: {len(list(d.iterdir()))} images")

params_complete = {
    "ged_timeout": 5,
    "skeletonization_threshold": 110,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}

results_complete, y_true_complete, y_pred_complete, run_dir_complete = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params_complete,
    sample_fraction=1.0,
    description="Experiment: manifest 'complete' images only (curated, resized 100x100)",
    local_path_template=str(_filtered_root / "{cls}"),
    nuclio_volume_path_template="/opt/nuclio/shared_storage/manifest_complete/{cls}",
)

In [ ]:
import optuna
import mlflow
from sklearn.metrics import accuracy_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Configuration ──────────────────────────────────────────
N_TRIALS = 200
N_STARTUP_TRIALS = 15
SAMPLE_FRACTION = 0.25
STUDY_NAME = "naturalagi-hp-tuning-ged-narrowed"

OPTUNA_MLFLOW_EXPERIMENT = STUDY_NAME

# Fixed feature set — converged across both GED and FGW studies
# (100% top-quartile consensus, vertical_direction harmful)
_FIXED_FEATURES = [
    "normalized_x", "normalized_y",
    "horizontal_direction", "cycle_count",
    "angle_with_ox",
]

def objective(trial: optuna.Trial) -> float:
    # Narrowed ranges based on dead-zone analysis from 500-trial GED study
    ged_timeout = trial.suggest_float("ged_timeout", 4.0, 15.0)
    skel_threshold = trial.suggest_int("skel_threshold", 90, 150, step=10)
    simplification_epsilon = trial.suggest_float("simplification_epsilon", 2.0, 7.0)

    # Delta parameterization: guarantees MINOR < GENERAL < SEVERE
    cost_minor = trial.suggest_float("cost_minor", 0.05, 0.4)
    gap_mg = trial.suggest_float("gap_minor_general", 0.05, 0.3)
    gap_gs = trial.suggest_float("gap_general_severe", 0.05, 0.3)
    cost_general = cost_minor + gap_mg
    cost_severe = cost_general + gap_gs

    cost_no_match = trial.suggest_float("cost_no_match", 0.8, 1.8)
    cost_impossible = trial.suggest_float("cost_impossible", 2.0, 50.0, log=True)

    params = {
        "comparison_method": "ged",
        "ged_timeout": ged_timeout,
        "skeletonization_threshold": skel_threshold,
        "simplification_epsilon": simplification_epsilon,
        "features": _FIXED_FEATURES,
        "property_normalizers": {
            "normalized_x": trial.suggest_float("norm_x", 1.0, 5.0),
            "normalized_y": trial.suggest_float("norm_y", 0.5, 4.5),
            "horizontal_direction": trial.suggest_float("norm_hdir", 0.5, 4.0),
            "vertical_direction": 3.0,  # inert — not in feature set
            "cycle_count": trial.suggest_float("norm_cycle", 0.5, 6.0),
            "angle_with_ox": trial.suggest_float("norm_angle", 150.0, 360.0),
        },
        "node_costs": {
            "NO_COST": 0.0,
            "MINOR": cost_minor,
            "GENERAL": cost_general,
            "SEVERE": cost_severe,
            "NO_MATCH": cost_no_match,
            "IMPOSSIBLE": cost_impossible,
        },
    }

    _, y_true, y_pred, _ = test_mnist_all(
        classes=list(classes_to_subclasses.keys()),
        params=params,
        sample_fraction=SAMPLE_FRACTION,
        description=f"optuna trial {trial.number}",
        quiet=True,
    )
    return accuracy_score(y_true, y_pred)


# ── Redirect MLflow to separate experiment ─────────────────
_original_experiment = evaluation.MLFLOW_EXPERIMENT
evaluation.MLFLOW_EXPERIMENT = OPTUNA_MLFLOW_EXPERIMENT

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5050"))
mlflow.set_experiment(OPTUNA_MLFLOW_EXPERIMENT)

try:
    with mlflow.start_run(run_name=f"optuna_{STUDY_NAME}_{N_TRIALS}trials") as parent_run:
        mlflow.log_params({
            "n_trials": N_TRIALS,
            "n_startup_trials": N_STARTUP_TRIALS,
            "sample_fraction": SAMPLE_FRACTION,
            "fixed_features": str(_FIXED_FEATURES),
            "sampler": "CmaEsSampler",
        })

        _storage = optuna.storages.RDBStorage(
            f"sqlite:///experiments/{STUDY_NAME}.db"
        )
        study = optuna.create_study(
            study_name=STUDY_NAME,
            storage=_storage,
            load_if_exists=True,
            direction="maximize",
            sampler=optuna.samplers.CmaEsSampler(
                seed=42,
                n_startup_trials=N_STARTUP_TRIALS,
                with_margin=True,
            ),
        )
        completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        remaining = max(0, N_TRIALS - completed)
        print(f"Resuming study: {completed} trials done, {remaining} remaining.")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)

        mlflow.log_metric("best_accuracy", study.best_value)
        mlflow.log_params({f"best.{k}": v for k, v in study.best_params.items()})

        try:
            importances = optuna.importance.get_param_importances(study)
            for param, imp in importances.items():
                mlflow.log_metric(f"importance.{param}", imp)
        except Exception:
            importances = {}

    print(f"\n{'='*50}")
    print(f"Best accuracy: {study.best_value*100:.2f}%")
    print(f"Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    print(f"{'='*50}")

    if importances:
        print("\nParameter importance:")
        for param, imp in importances.items():
            bar = "█" * int(imp * 30)
            print(f"  {param:25s} {imp:.3f} {bar}")

    # ── Full-set validation of best params ─────────────────
    bp = study.best_params
    full_test_params = {
        "comparison_method": "ged",
        "ged_timeout": bp["ged_timeout"],
        "skeletonization_threshold": bp["skel_threshold"],
        "simplification_epsilon": bp["simplification_epsilon"],
        "features": _FIXED_FEATURES,
        "property_normalizers": {
            "normalized_x": bp["norm_x"],
            "normalized_y": bp["norm_y"],
            "horizontal_direction": bp["norm_hdir"],
            "vertical_direction": 3.0,
            "cycle_count": bp["norm_cycle"],
            "angle_with_ox": bp["norm_angle"],
        },
        "node_costs": {
            "NO_COST": 0.0,
            "MINOR": bp["cost_minor"],
            "GENERAL": bp["cost_minor"] + bp["gap_minor_general"],
            "SEVERE": bp["cost_minor"] + bp["gap_minor_general"] + bp["gap_general_severe"],
            "NO_MATCH": bp["cost_no_match"],
            "IMPOSSIBLE": bp["cost_impossible"],
        },
    }

    print(f"\n{'='*50}")
    print("Running full-set validation...")
    print(json.dumps(full_test_params, indent=2))

    results, y_true, y_pred, run_dir = test_mnist_all(
        classes=list(classes_to_subclasses.keys()),
        params=full_test_params,
        sample_fraction=1.0,
        description=f"Full test with best Optuna params (study: {STUDY_NAME}, proxy: {study.best_value*100:.2f}%)",
    )

    full_accuracy = accuracy_score(y_true, y_pred)
    print(f"\nFull-dataset accuracy: {full_accuracy*100:.2f}%")
    print(f"Optuna proxy accuracy (25%): {study.best_value*100:.2f}%")
    print(f"Delta: {(full_accuracy - study.best_value)*100:+.2f}pp")

finally:
    evaluation.MLFLOW_EXPERIMENT = _original_experiment

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

delete_test_neo4j_nodes()

class_number = 1
img_num = 448
image_id = f"mnist_{class_number}_{img_num:05d}"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
nuclio_path = f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"

img_file = f"{image_id}.png"
img_local = os.path.join(local_path, img_file)
if os.path.exists(img_local):
    img = mpimg.imread(img_local)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis("off")
    plt.show()

image_id_for_test = str(uuid.uuid4())
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "image_id": image_id_for_test,
    "session_id": "test",
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(nuclio_path, img_file), params=params, timeout=60)
print(json.dumps(result, indent=2))

In [ ]:
# remove_concept("3_1")
retrain_concept(number=2, subclass=2, with_concept_creation=True)